# 08 · Databricks SQL: consultas y gestión de datos

⏱ **15 min** · Módulo 6 · Slide de la Intro: 24 (consultas y gestión de datos)

**Objetivo:** hacer sobre el SQL warehouse el trabajo del día a día: consultar, crear y modificar tablas, dar permisos y mantener el rendimiento.

> **Este notebook es SQL puro y corre en el SQL warehouse.** Arriba a la derecha, en el selector de cómputo, elige
> **SQL warehouses → Serverless Starter Warehouse** (no *Serverless*).

| Paso | Qué haces |
|---|---|
| 1 | Entrar al esquema y comprobar dónde estoy |
| 2 | La consulta del slide |
| 3 | Más consultas de analista |
| 4 | Crear una tabla y una vista |
| 5 | Modificar una tabla |
| 6 | Dar permisos con `GRANT` |
| 7 | Mantener el rendimiento |

**Requiere:** notebook 05.

In [0]:
USE SCHEMA lab_tvmaze_m56

## Paso 1 · ¿Dónde estoy?

In [0]:
SELECT current_catalog() AS catalogo, current_schema() AS esquema, current_user() AS usuario,
       current_version().dbsql_version AS version_del_warehouse

👀 **Qué observar:** `version_del_warehouse` solo tiene valor cuando la consulta corre en un SQL warehouse.
Si sale vacía, el notebook está conectado al cómputo *Serverless*: cámbialo arriba a la derecha.

## Paso 2 · La consulta del slide

`SELECT region, SUM(monto) AS total FROM ventas GROUP BY region ORDER BY total DESC`, con nuestros datos.

In [0]:
SELECT network_name, SUM(episodes) AS total
FROM gold_network_daily
GROUP BY network_name
ORDER BY total DESC

👀 **Qué observar:** en el resultado, el botón **+** permite agregar una visualización sin salir del notebook.

## Paso 3 · Más consultas de analista

In [0]:
-- ¿Cuántos episodios y minutos hay por día?
SELECT airdate, SUM(episodes) AS episodios, SUM(total_minutes) AS minutos
FROM gold_network_daily
GROUP BY airdate
ORDER BY airdate

In [0]:
-- ¿Cuál fue la red con más episodios cada día?
SELECT airdate, network_name, episodes
FROM gold_network_daily
QUALIFY ROW_NUMBER() OVER (PARTITION BY airdate ORDER BY episodes DESC) = 1
ORDER BY airdate

👀 **Qué observar:** el analista trabaja sobre gold, la tabla que el Job mantiene al día. No necesita saber nada de bronze ni de silver.

## Paso 4 · Crear una tabla y una vista

Gestión de datos, primera parte: crear.

In [0]:
CREATE OR REPLACE TABLE resumen_diario AS
SELECT airdate, SUM(episodes) AS episodios, COUNT(*) AS redes, SUM(total_minutes) AS minutos
FROM gold_network_daily
GROUP BY airdate

In [0]:
CREATE OR REPLACE VIEW top_redes AS
SELECT network_name, SUM(episodes) AS episodios
FROM gold_network_daily
GROUP BY network_name
ORDER BY episodios DESC
LIMIT 10

In [0]:
SELECT * FROM top_redes

👀 **Qué observar:** la **tabla** guarda una copia del resultado (hay que volver a crearla para actualizarla);
la **vista** guarda solo la consulta y siempre lee el gold de ahora.

## Paso 5 · Modificar una tabla

Agregar una columna, llenarla y documentar la tabla.

In [0]:
ALTER TABLE resumen_diario ADD COLUMN es_fin_de_semana BOOLEAN

In [0]:
UPDATE resumen_diario SET es_fin_de_semana = dayofweek(airdate) IN (1, 7)

In [0]:
COMMENT ON TABLE resumen_diario IS 'Episodios, redes y minutos por día. Fuente: gold_network_daily.'

In [0]:
SELECT * FROM resumen_diario ORDER BY airdate

## Paso 6 · Dar permisos con `GRANT`

Los analistas reciben lo que necesitan: gold y el resumen. Nada más.

In [0]:
GRANT SELECT ON TABLE gold_network_daily TO `account users`

In [0]:
GRANT SELECT ON TABLE resumen_diario TO `account users`

In [0]:
SHOW GRANTS ON TABLE gold_network_daily

👀 **Qué observar:** es el `GRANT SELECT ON TABLE ventas TO analistas` del slide. Con el `USE SCHEMA` que dimos en el notebook 02,
el grupo ya puede consultar estas dos tablas; bronze, silver y `suscriptores` siguen cerradas.

## Paso 7 · Mantener el rendimiento

`OPTIMIZE` junta archivos pequeños en archivos más grandes; `ANALYZE` actualiza las estadísticas que usa el motor para planificar las consultas.

In [0]:
DESCRIBE DETAIL gold_network_daily

In [0]:
OPTIMIZE gold_network_daily

In [0]:
ANALYZE TABLE gold_network_daily COMPUTE STATISTICS

👀 **Qué observar:** en `DESCRIBE DETAIL`, `numFiles` vale 1: la tabla es un solo archivo, así que `OPTIMIZE` no tuvo nada que juntar
(`numFilesAdded` = 0). En una tabla que recibe cargas pequeñas todos los días, convierte cientos de archivos en unos pocos.
En las tablas administradas Databricks también lo hace por su cuenta (*predictive optimization*).
Para encontrar consultas lentas, abre el menú **Query History**: muestra cada consulta del warehouse con su duración.

## ✔ Checkpoint

- [ ] Ejecuté consultas en el SQL warehouse (`version_del_warehouse` tiene valor)
- [ ] Creé una tabla y una vista, y sé en qué se diferencian
- [ ] Modifiqué una tabla con `ALTER TABLE` y `UPDATE`
- [ ] Di permisos con `GRANT` y los revisé con `SHOW GRANTS`
- [ ] Sé para qué sirven `OPTIMIZE` y `ANALYZE`

**Siguiente:** `09_Todo_Conectado_y_Cierre`